# **Building a Hotel Recommendation Model**

## **Import Libraries**

In [19]:
import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from scipy.sparse.linalg import svds

from recommender_1 import CFRecommender # Custom class created

import warnings
warnings.filterwarnings("ignore")


---

## **Load Dataset**

In [3]:
df = pd.read_csv("C:/Users/aimee/OneDrive/Documents/ML_Projects/Spl_Mod_1_MLOps/Hotel_Recommender_System/hotels.csv")

hotel_df = df.copy()

In [4]:
hotel_df.shape

(40552, 8)

In [5]:
hotel_df.head()

,travelCode,userCode,name,place,days,price,total,date
0,0,0,Hotel A,Florianopolis (SC),4,313.02,1252.08,09/26/2019
1,2,0,Hotel K,Salvador (BH),2,263.41,526.82,10/10/2019
2,7,0,Hotel K,Salvador (BH),3,263.41,790.23,11/14/2019
3,11,0,Hotel K,Salvador (BH),4,263.41,1053.64,12/12/2019
4,13,0,Hotel A,Florianopolis (SC),1,313.02,313.02,12/26/2019


In [6]:
hotel_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40552 entries, 0 to 40551
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   travelCode  40552 non-null  int64  
 1   userCode    40552 non-null  int64  
 2   name        40552 non-null  object 
 3   place       40552 non-null  object 
 4   days        40552 non-null  int64  
 5   price       40552 non-null  float64
 6   total       40552 non-null  float64
 7   date        40552 non-null  object 
dtypes: float64(2), int64(3), object(3)
memory usage: 2.5+ MB


---

## **Advanced Preprocessing, Encoding and Manipulation**

#### **Filter Active Users**

Keep users with at least 2 interactions.

In [7]:
user_interactions = hotel_df.groupby(['userCode','name']).size().groupby('userCode').size()

active_users = user_interactions[user_interactions >= 2].reset_index()[['userCode']]

interactions_df = hotel_df.merge(active_users, on='userCode', how='right')

In [8]:
interactions_df.shape

(40524, 8)

**Create Implicit Feedback**

In [9]:
interactions_df['interaction'] = 1

#### **Encode Hotel Names**

Convert hotel names into numeric IDs

In [10]:
label_encoder = LabelEncoder()
interactions_df['name_encoded'] = label_encoder.fit_transform(interactions_df['name'])

#### **Aggregate Interactions**

In [11]:
interactions_full_df = interactions_df.groupby(
    ['userCode', 'name_encoded']
)['interaction'].sum().reset_index()

---

## **Train-Test Split**

In [12]:
train_df, test_df = train_test_split(
    interactions_full_df,
    stratify=interactions_full_df['userCode'],
    test_size=0.25,
    random_state=42
)

---

## **Matrix Creation and Factorization**

#### **Create User-Item Matrix**

Rows = Users, Columns = Hotels

In [13]:
pivot_df = train_df.pivot(
    index='userCode',
    columns='name_encoded',
    values='interaction'
).fillna(0)

matrix = pivot_df.values
user_ids = list(pivot_df.index)

**Normalize Matrix (Improves SVD)**

In [14]:
scaler = MinMaxScaler()
matrix_scaled = scaler.fit_transform(matrix)

#### **Matrix Factorization (SVD)**

In [17]:
NUM_FACTORS = 8  # Try 10, 20, 50

U, sigma, Vt = svds(matrix_scaled, k=NUM_FACTORS)
sigma = np.diag(sigma)

all_user_predicted_ratings = np.dot(np.dot(U, sigma), Vt)

#### **Create Prediction DataFrame**

In [18]:
cf_preds_df = pd.DataFrame(
    all_user_predicted_ratings,
    columns=pivot_df.columns,
    index=user_ids
).transpose()

---

## **Build Recommender Class and Model Building**

In [14]:
# class CFRecommender:
#     def __init__(self, predictions_df, items_df):
#         self.predictions_df = predictions_df
#         self.items_df = items_df

#     def recommend_items(self, user_id, topn=5):
#         if user_id not in self.predictions_df.columns:
#             raise KeyError("User not found")

#         sorted_preds = self.predictions_df[user_id] \
#             .sort_values(ascending=False) \
#             .reset_index() \
#             .rename(columns={user_id: 'score'})

#         recommendations = sorted_preds.head(topn)

#         recommendations = recommendations.merge(
#             self.items_df,
#             on='name_encoded'
#         )[['name', 'score']]

#         return recommendations

#### **Initialize Model**

In [21]:
recommender = CFRecommender(cf_preds_df, interactions_df)

#### **Popularity Baseline**

In [22]:
popularity_df = train_df.groupby('name_encoded')['interaction'] \
    .sum().sort_values(ascending=False).reset_index()

popularity_df = popularity_df.merge(
    interactions_df[['name_encoded','name']],
    on='name_encoded'
).drop_duplicates()

#### **Evaluation Metrics**

In [23]:
def precision_recall_at_k(model, train_df, test_df, k=5):
    precisions = []
    recalls = []

    test_user_items = test_df.groupby('userCode')['name_encoded'].apply(set)

    for user in test_user_items.index:

        if user not in model.predictions_df.columns:
            continue

        true_items = test_user_items[user]

        seen_items = train_df[train_df['userCode']==user]['name_encoded'].tolist()

        preds = model.predictions_df[user] \
            .sort_values(ascending=False)

        preds = preds[~preds.index.isin(seen_items)].head(k).index

        pred_items = set(preds)

        hits = len(true_items & pred_items)

        precision = hits / k
        recall = hits / len(true_items) if len(true_items) > 0 else 0

        precisions.append(precision)
        recalls.append(recall)

    return np.mean(precisions), np.mean(recalls)

#### **Evaluate Model**

In [24]:
precision5, recall5 = precision_recall_at_k(recommender, train_df, test_df, k=5)
precision10, recall10 = precision_recall_at_k(recommender, train_df, test_df, k=10)

print(f"Precision@5: {precision5:.4f}")
print(f"Recall@5: {recall5:.4f}")

print(f"Precision@10: {precision10:.4f}")
print(f"Recall@10: {recall10:.4f}")

Precision@5: 0.3742
Recall@5: 0.9837
Precision@10: 0.1887
Recall@10: 1.0000


#### **Evaluate Popularity Baseline**

In [25]:
def popularity_precision_at_k(train_df, test_df, k=5):
    popular_items = train_df.groupby('name_encoded')['interaction'] \
        .sum().sort_values(ascending=False).head(k).index

    precisions = []

    test_user_items = test_df.groupby('userCode')['name_encoded'].apply(set)

    for user in test_user_items.index:
        true_items = test_user_items[user]
        hits = len(set(popular_items) & true_items)
        precision = hits / k
        precisions.append(precision)

    return np.mean(precisions)

pop_precision = popularity_precision_at_k(train_df, test_df, k=5)

print(f"Popularity Precision@5: {pop_precision:.4f}")

Popularity Precision@5: 0.2294


#### **Save Model**

In [26]:
import pickle

with open('recommender_1.pkl', 'wb') as f:
    pickle.dump((recommender, cf_preds_df, interactions_df), f)